# R4 — 패러프레이즈 통제군 · arm2 기하 · 기하 궤적

R3에서 드러난 구멍 셋을 닫는다. 상세 = `../R3_STATUS.md` §4.

- ① **arm0·arm2 × `agreeable_para1`이 미측정** — 패러프레이즈 결과에 통제군이 없어서
  "para1이 그냥 더 센 프롬프트일 뿐"이라는 대안 설명이 열려 있다
- ② **arm2에 프로브가 없다** — 일반 파인튜닝의 기하가 없으면 기하 서명이
  페르소나 학습 고유라고 말할 수 없다
- ③ **기하의 라운드 궤적이 없다** — 시드42 프로브 11개가 전부
  `probe_r2`·κ_harm≡1.0으로 무효라 종점 2점만 있다

셀 0~8은 `nb_seeds.ipynb`와 같다(셀 8에 `probe_rounds`와 `run_baseline(mmlu=)`만 추가).
★**이 파일을 직접 수정하지 말 것** — `src/make_nb_r4.py`가 생성한다.

## 0. Secrets

⚠️ **Secrets는 계정이 아니라 노트북마다 붙는다.** 새 노트북을 만들거나 업로드하면 계정에 등록돼
있어도 안 딸려온다 → **Add-ons → Secrets → `HF_TOKEN` 토글 ON.**

In [ ]:
import os, torch

# 가속기는 시크릿과 성질이 같다 — **노트북마다** 붙고, Import하면 None으로 초기화된다.
# 여기서 안 막으면 패키지 설치를 다 하고 unsloth 내부 스택에서 죽는다.
if not torch.cuda.is_available():
    raise SystemExit(
        "\n[GPU 없음] 이 세션에 가속기가 안 붙었다. unsloth는 CPU에서 로드 자체가 안 된다.\n"
        "  1) 우측 Settings -> Accelerator -> GPU T4 x2 (또는 P100) 선택 후 다시 Run.\n"
        "     노트북을 새로 Import하면 가속기 설정은 따라오지 않는다.\n"
        "  2) 이미 GPU로 돼 있다면 **주간 할당량(30h) 소진**이다 — 우측 상단 게이지를 볼 것.\n"
        "     소진되면 Kaggle은 경고 없이 CPU로 돌린다.\n"
        f"  torch={torch.__version__} cuda_built={torch.version.cuda}"
    )
print(f"GPU: {torch.cuda.device_count()}x {torch.cuda.get_device_name(0)}")

from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise SystemExit(
        "\n[HF_TOKEN 없음] Secrets는 계정이 아니라 **노트북마다** 붙여야 한다.\n"
        "  상단 메뉴 Add-ons -> Secrets -> HF_TOKEN 토글 ON -> 다시 Run\n"
        "  (계정에 이미 등록돼 있어도 새 노트북에는 자동으로 안 딸려온다)\n"
        f"  원본 오류: {e}"
    )
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 1. 패키지 설치 (매 세션 필수)
Unsloth 공식 Llama3.1(8B) 노트북 핀: `transformers==4.56.2`, `trl==0.22.2`.

In [ ]:
import subprocess, sys

def pip(*args):
    """`!pip` 매직 대신 subprocess — 이 노트북의 모든 셀이 순수 파이썬이라
    업로드 전에 compile()로 문법 검증을 걸 수 있다(RUNBOOK 함정 15)."""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("unsloth")
pip("transformers==4.56.2", "trl==0.22.2")
pip("git+https://github.com/dsbowen/strong_reject.git@main")
pip("lm-eval")
print("packages ok")

## 2. 실험 폴더 + 학습 데이터 복구
데이터는 1차 것을 **그대로 재사용**한다 — 다시 만들면 1차와의 비교가 끊긴다.

In [ ]:
import os, shutil, sys, glob

EXP_ROOT  = "/kaggle/working/experiment"
CKPT_ROOT = "/kaggle/tmp/ckpt"   # 휘발성 대용량 (머지 모델 16GB/개)

def _input_tree():
    """붙은 데이터셋을 3단계까지 나열 — 경로가 안 맞을 때 진단용."""
    seen = sorted(glob.glob("/kaggle/input/*")
                  + glob.glob("/kaggle/input/*/*")
                  + glob.glob("/kaggle/input/*/*/*"))
    return seen or ["(비어 있음 — 붙은 데이터셋이 하나도 없다)"]

# 코드 데이터셋 위치 자동 탐색. 마운트 경로가 /kaggle/input/<slug>/ 이기도 하고
# /kaggle/input/datasets/<owner>/<slug>/ 이기도 해서 하드코딩하지 않는다.
if not os.path.exists(EXP_ROOT):
    hits = glob.glob("/kaggle/input/**/src/eval_refusal.py", recursive=True)
    if not hits:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[코드 데이터셋 없음] 'branch-1-premise'가 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> branch-1-premise / gen-data-backup 둘 다 추가\n"
            "  (Secrets와 마찬가지로 데이터셋도 노트북마다 따로 붙여야 한다)"
        )
    INPUT_DIR = os.path.dirname(os.path.dirname(hits[0]))
    print("[code] 데이터셋 발견:", INPUT_DIR)
    shutil.copytree(INPUT_DIR, EXP_ROOT)
os.makedirs(CKPT_ROOT, exist_ok=True)
os.chdir(EXP_ROOT)
sys.path.insert(0, os.path.join(EXP_ROOT, "src"))

# zip(압축)과 이미 풀린 폴더(Kaggle 자동 압축해제) 둘 다 지원
if not os.path.exists("data/arm1/round1.jsonl"):
    zip_cands = (glob.glob("/kaggle/input/**/gen_data_backup.zip", recursive=True)
                 + glob.glob("/kaggle/working/**/gen_data_backup.zip", recursive=True))
    dir_cands = glob.glob("/kaggle/input/**/arm1/round1.jsonl", recursive=True)
    if zip_cands:
        shutil.unpack_archive(zip_cands[0], "data")
        print("[data] 복구(zip):", zip_cands[0])
    elif dir_cands:
        src_root = os.path.dirname(os.path.dirname(dir_cands[0]))
        shutil.copytree(src_root, "data", dirs_exist_ok=True)
        print("[data] 복구(폴더):", src_root)
    else:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[학습 데이터 없음] 'gen-data-backup'이 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> gen-data-backup 추가"
        )

print("cwd:", os.getcwd())
print("src:", sorted(os.listdir("src")))
print("data arms:", sorted(os.path.basename(d) for d in glob.glob("data/arm*")))

# R2 코드가 맞는지 게이트 — 하나라도 실패하면 1차 코드가 올라가 있는 것이다
for path, needle in [("src/eval_refusal.py", "schema"),
                     ("src/train_round.py", "frac_ckpt"),
                     ("src/probe_directions.py", "harm_frame"),
                     ("configs/arms.json", "eval_lenses")]:
    assert os.path.exists(path), f"R2 코드 아님: {path} 없음 → Dataset New Version 재업로드 필요"
    if needle:
        assert needle in open(path, encoding="utf-8").read(), \
            f"R2 코드 아님: {path}에 '{needle}' 없음 → Dataset New Version 재업로드 필요"
print("[gate] R2 코드 확인됨")

## 3. 세션 함수 정의

In [ ]:
import json, shutil, subprocess, os

BASE = "unsloth/Meta-Llama-3.1-8B-Instruct"
FULL = " --full"       # R2는 전 라운드 n=313. 1차의 n 불일치가 라운드 비교를 막았다.
MMLU_LIMIT = "30"

def run(cmd: str):
    print(f"\n$ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f"실패: {cmd}"

def check(path, min_kb=20):
    """per-prompt 레코드가 실제로 쓰였는지 즉시 확인 — 1차의 핵심 실패를 막는 게이트."""
    kb = os.path.getsize(path) / 1024
    print(f"[check] {os.path.basename(path)} = {kb:.0f} KB")
    assert kb > min_kb, f"per-prompt 레코드가 비었다: {path}"

def probe_baseline(seed, harm_frame="judgement"):
    """동결기저(θ_0, ∅). 학습 전에 한 번. 이후 프로브가 이걸 기준으로 읽는다.
       ⚠️기저와 이후 라운드의 harm_frame이 다르면 ρ·D가 무의미해진다 — 같은 값을 쓸 것."""
    run(f"python src/probe_directions.py --model_path {BASE} --arm arm0 --round 0 --seed {seed} "
        f"--harm_frame {harm_frame}")
    return f"results/probe_arm0_r0_s{seed}.json"

def run_session(arm, seed, lenses="none,agreeable,principled",
                probe=False, frac=False, mmlu=True, eval_rounds=(1, 2, 3),
                harm_frame="judgement", mmlu_limit=None, free_ckpt=True,
                probe_rounds=None):
    """★2026-07-29 확장 3건:
       eval_rounds  — 학습은 순차라 건너뛸 수 없지만 **평가는 건너뛸 수 있다.**
                      비용의 거의 전부가 eval(1렌즈 n=313 = ~1,800s)이므로 라운드3만
                      재면 세션이 ~3.4h 짧아진다. 시드 반복은 라운드별 곡선이 아니라
                      최종 효과만 필요하므로 eval_rounds=(3,)이 맞다.
       harm_frame   — 프로브 v_harm 추출 프레이밍. 기본 judgement(고친 것).
                      raw는 S1·S2 재현 전용이며 κ_harm≡1이라 무정보다.
       mmlu_limit   — None이면 전역 MMLU_LIMIT. 논문이 보고하는 대조는 **동일 570문항
                      (limit=10)** 이므로 새 arm은 10으로 도는 편이 비교 가능하고 3배 싸다.
       probe_rounds — ★R4 신설. 프로브가 `r in eval_rounds`에 묶여 있어서
                      **eval 없이 프로브만** 뜰 수 없었다. 기하의 라운드 궤적을 재려면
                      eval(렌즈당 25분)을 다시 돌릴 이유가 없다 — 시드42 eval은 이미 있다.
                      None이면 eval_rounds를 따른다(기존 동작 그대로)."""
    base_dirs = probe_baseline(seed, harm_frame) if probe else None
    pr_rounds = eval_rounds if probe_rounds is None else probe_rounds
    prev, prev_dir = BASE, None

    for r in [1, 2, 3]:
        ckpt = f"{CKPT_ROOT}/{arm}_r{r}_s{seed}"
        frac_arg = " --frac_ckpt 0.25,0.5" if (frac and r == 1) else ""
        run(f"python src/train_round.py --arm {arm} --round {r} --seed {seed} "
            f"--base_model {prev} --data data/{arm}/round{r}.jsonl "
            f"--output_dir {ckpt}{frac_arg}")

        # 세분 라운드: 라운드1 학습 중 저장된 어댑터를 원본 BASE 위에 얹어 평가
        if frac and r == 1:
            for f in ["0p25", "0p5"]:
                ad = f"{CKPT_ROOT}/{arm}_s{seed}/adapter_frac{f}"
                if not os.path.exists(ad):
                    print(f"[warn] 세분 체크포인트 없음: {ad}")
                    continue
                rn = "0.25" if f == "0p25" else "0.5"
                run(f"python src/eval_refusal.py --model_path {BASE} --adapter_path {ad} "
                    f"--arm {arm} --round {rn} --seed {seed} --lenses {lenses}{FULL}")
                check(f"results/eval_{arm}_r{f}_s{seed}.json")
                if probe:
                    run(f"python src/probe_directions.py --model_path {BASE} --adapter_path {ad} "
                        f"--arm {arm} --round {rn} --seed {seed} --baseline_dirs {base_dirs}")

        if r in eval_rounds:
            run(f"python src/eval_refusal.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --lenses {lenses}{FULL}")
            check(f"results/eval_{arm}_r{r}_s{seed}.json")
        else:
            print(f"[skip] eval arm={arm} r={r} — eval_rounds={eval_rounds} (비용 절감)")

        if probe and r in pr_rounds:
            run(f"python src/probe_directions.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --baseline_dirs {base_dirs} --harm_frame {harm_frame}")

        if mmlu and r == 3:
            run(f"bash src/run_mmlu.sh {ckpt} {arm} {r} {mmlu_limit or MMLU_LIMIT}")

        # 디스크: 머지본 1개 = 16GB. 다음 라운드 전에 이전 것 삭제.
        if prev_dir and os.path.exists(prev_dir):
            shutil.rmtree(prev_dir)
            print(f"[disk] 삭제: {prev_dir}")
        prev, prev_dir = ckpt, ckpt

    # 루프는 '이전' 라운드만 지운다 — 마지막 라운드 머지본 16GB가 그대로 남는다.
    # 한 세션만 돌면 무해하지만, 세션을 이어 돌리면 세션당 16GB씩 쌓여
    # 시간이 아니라 디스크가 먼저 터진다. MMLU는 위 루프 안에서 이미 끝났다.
    if free_ckpt and prev_dir and os.path.exists(prev_dir):
        shutil.rmtree(prev_dir)
        print(f"[disk] 삭제(최종): {prev_dir}")
        return None
    return prev

def run_baseline(seed, lenses="none,agreeable,principled", mmlu_limit=None,
                 mmlu=True):
    """arm0 = 원본 모델. 학습 없음. 페르소나 방아쇠 자체가 무해함을 보이는 통제군.
       mmlu — ★R4 신설. arm0의 limit=10 MMLU는 이미 있다. 다시 돌리면 20분을 버리고
              results/mmlu_arm0_r0/ 에 타임스탬프 파일이 하나 더 쌓인다."""
    run(f"python src/eval_refusal.py --model_path {BASE} --arm arm0 --round 0 "
        f"--seed {seed} --lenses {lenses}{FULL}")
    check(f"results/eval_arm0_r0_s{seed}.json")
    if mmlu:
        run(f"bash src/run_mmlu.sh {BASE} arm0 0 {mmlu_limit or MMLU_LIMIT}")
    else:
        print("[skip] MMLU arm0 — 이미 측정됨(limit=10)")

print("정의 완료")

## 4. ★이번 세션 — `BATCH` 하나만 바꾼다

| BATCH | 세션 | 무엇 | 닫는 것 | 예상 |
|---|---|---|---|---|
| ~~**1**~~ | S8 `arm0` · S9 `arm2` | 4렌즈(+`agreeable_para1`) + arm2 프로브 | ① 패러프레이즈 통제군 · ② arm2 기하 | ✅완료 4.0h |
| **2** | S10 `arm1` · S11 `arm3` | 프로브 r1/r2/r3 + **r3 eval 4렌즈** + MMLU(limit=10) | ③ 궤적 · ④ 능력유지 | ~5~5.5h |

⚠️**BATCH 2는 `eval_arm{1,3}_r3_s42.json`을 덮어쓴다.** 4렌즈 상위집합이지만
**재학습된 체크포인트라 값이 달라진다** — BATCH 1의 arm2가 그랬다(`unsloth 7.5→7.6`,
`none` 0.818→0.866). **이건 의도된 것이다**: 궤적 프로브와 행동 수치가 같은 모델에서
나와야 비교가 성립한다. 시드42 `para1`도 새로 생겨 para1이 3시드가 된다.

⚠️★**BATCH 2 후 로컬에서 `python src/sample_m03.py --seed 42 --force`를 돌릴 것** —
`EXP-M03` 표본이 이 파일들에서 층을 뽑으므로 무효가 된다. 라벨링 전이면 비용 0이다.

⚠️게이트는 **학습 유무로 요구 수준을 나눈다**: 무학습 arm(arm0)은 거부 개수가 정확히
재현되어야 하고(어기면 실패), 재학습 arm은 달라져도 **경고만** 한다(교훈 22).

In [ ]:
import json

# ── 리비전 게이트 ────────────────────────────────────────────────────────────
# R3 BATCH 2에서 **낡은 노트북**이 옛 TAG로 zip을 냈다(내용은 s2718인데 이름은 s1337).
# 리포의 셀 12는 이미 고쳐져 있었고 Kaggle 쪽만 낡았던 것이다 — 사람이 눈으로 비교해서는
# 못 잡는다. Dataset(configs/arms.json)과 노트북에 같은 리비전을 박고 여기서 대조한다.
NB_NAME = "nb_r4"
NB_REV = "R4b"
_ds_revisions = json.load(open("configs/arms.json", encoding="utf-8")).get("nb_revisions", {})
_ds_rev = _ds_revisions.get(NB_NAME)
assert _ds_rev == NB_REV, (
    f"리비전 불일치: 노트북={NB_NAME}/{NB_REV} Dataset={_ds_rev}. "
    f"둘 중 하나가 낡았다 — Dataset을 다시 올리거나 노트북을 다시 Import하라")
print(f"[gate] 리비전 {NB_NAME}/{NB_REV} 일치")

# ★①②의 렌즈. principled까지 넣는 이유는 기존 파일을 덮어써도 상위집합이 되게 하려는 것.
LENSES_CONTROL = "none,agreeable,principled,agreeable_para1"

# ★★ 커밋마다 이 값 하나만 바꾼다. ★★
# 줄을 주석 처리해서 고르지 않는다 — 셀 부분 편집으로 줄을 잘못 살리는 것이
# 이 프로젝트에서 이미 나온 실패 방식이다.
BATCH = 1

BATCHES = {
    # ① 패러프레이즈 통제군 + ② arm2 기하. arm0은 학습이 없어 eval만 돈다.  ✅완료
    1: [("S8", "arm0", 42), ("S9", "arm2", 42)],
    # ③ 기하의 라운드 궤적 + r3 eval 재측정(자기정합) + ④ MMLU limit=10.
    2: [("S10", "arm1", 42), ("S11", "arm3", 42)],
}
assert BATCH in BATCHES, f"BATCH는 {sorted(BATCHES)} 중 하나여야 한다"
PLAN = BATCHES[BATCH]
print(f"R4 BATCH {BATCH}: " + ", ".join(f"{n}({a} x {s})" for n, a, s in PLAN))


def snap(tag):
    """세션이 끝날 때마다 zip을 떠 둔다. results/는 누적되므로 마지막 zip 하나에
       그 커밋의 전부가 들어 있다. 뒤 세션이 죽어도 앞 세션은 output에 남는다."""
    import shutil, os
    zp = shutil.make_archive(f"/kaggle/working/r4_results_{tag}", "zip",
                             "/kaggle/working/experiment/results")
    print(f"[snap] {zp} | {os.path.getsize(zp)/1e6:.2f} MB", flush=True)


for name, arm, seed in PLAN:
    print(f"\n===== {name}: {arm} x seed {seed} =====", flush=True)
    if BATCH == 1:
        if arm == "arm0":
            # 학습 없음. MMLU도 없다(이미 측정됨). 4렌즈 x 25분 = 약 1.7h.
            run_baseline(seed, lenses=LENSES_CONTROL, mmlu=False)
        else:
            # 재학습 35분 + 4렌즈 1.7h + 프로브 5분. probe=True가 동결기저도 만든다.
            run_session(arm, seed, lenses=LENSES_CONTROL, eval_rounds=(3,),
                        probe=True, mmlu=False)
    else:
        # ③ 궤적 프로브 r1/r2/r3 + **r3 eval을 같이 돌린다.**
        #   재학습하면 unsloth 버전 차이로 R3 체크포인트와 달라진다(BATCH 1의 arm2가 그랬다).
        #   궤적 프로브와 행동 수치가 **같은 모델**에서 나오게 하려면 r3를 다시 재야 한다.
        #   4렌즈로 도는 이유: 기존 3렌즈 파일의 상위집합이 되고, 시드42 para1이 새로 생겨
        #   arm1·arm3의 para1이 3시드가 된다. mmlu_limit=10은 R4 ④를 같이 닫는다.
        run_session(arm, seed, lenses=LENSES_CONTROL, eval_rounds=(3,),
                    probe=True, probe_rounds=(1, 2, 3),
                    mmlu=True, mmlu_limit="10")
    snap(f"{name.lower()}_{arm}_s{seed}")


## 5. ★결과 회수

In [ ]:
import shutil, os, glob

# TAG는 BATCH에서 유도한다 — 손으로 적으면 배치 내용과 어긋난다.
# R3 BATCH 2가 정확히 그렇게 틀렸다(낡은 노트북의 하드코딩된 TAG).
TAG = f"batch{BATCH}"

# snap()이 세션마다 떠 둔 것과 별개인 **그 커밋의 최종본**이다. results/는 누적이므로
# 이 zip 하나에 배치 전부가 들어 있다 — 로컬로 내릴 건 이것 하나면 된다.
zp = shutil.make_archive(f"/kaggle/working/r4_results_{TAG}", "zip",
                         "/kaggle/working/experiment/results")
print("zip:", zp, "| size MB:", round(os.path.getsize(zp)/1e6, 2))
for p in sorted(glob.glob("/kaggle/working/experiment/results/*.json")):
    print(f"  {os.path.getsize(p)/1024:7.0f} KB  {os.path.basename(p)}")


## 6. 세션이 성공했는지

BATCH에 따라 보는 것이 다르다. 여기서 걸려도 `snap()`이 zip을 이미 떠 뒀다 — 회수는 된다.

In [ ]:
import json, glob, os, re

N = 313
# (arm, round) -> {"trained": 이 배치에서 재학습되는가, "counts": {lens: 기존 거부 개수}}
#
# ★2026-07-30 재설계 (교훈 22). 원래는 모든 arm에 |d| <= 10을 요구했는데, R4 BATCH 1에서
#   arm2가 +15로 걸려 배치가 죽었다. 원인은 이상이 아니라 **unsloth 2026.7.5 -> 7.6**이었다:
#   같은 시드로 재학습해도 커널 수준 부동소수점이 달라 3라운드 누적되면 15문항이 뒤집힌다.
#   같은 실행에서 **arm0(학습 없음)는 292/294/309를 개수까지 정확히 재현**했다.
#   => 학습이 없는 arm에만 결정성을 요구한다. 재학습 arm에 비트 재현성을 요구하면
#      게이트가 정상 실행을 죽인다(실제로 죽였다).
KNOWN = {
    ("arm0", 0.0): {"trained": False,
                    "counts": {"none": 292, "agreeable": 294, "principled": 309}},
    ("arm2", 3.0): {"trained": True,
                    "counts": {"none": 271, "agreeable": 295, "principled": 310}},
    ("arm1", 3.0): {"trained": True,
                    "counts": {"none": 306, "agreeable": 183, "principled": 299}},
    ("arm3", 3.0): {"trained": True,
                    "counts": {"none": 310, "agreeable": 174, "principled": 287}},
}


def read_probe(arm, rtag, seed):
    pp = f"results/probe_{arm}_r{rtag}_s{seed}.json"
    assert os.path.exists(pp), f"프로브가 없다: {pp}"
    pr = json.load(open(pp, encoding="utf-8"))
    kr = pr["readings"]["kappa_v_ref"]
    kh = pr["readings"]["kappa_v_harm"]
    assert pr.get("schema") == "probe_r3", (
        f"{pp}: 옛 프로브 스키마 — Dataset을 새로 올려야 한다")
    assert abs(kh - 1.0) > 1e-4, (
        f"{pp}: K_harm이 1.0 — S1,S2를 무효화한 결함이 재발했다")
    return kr, kh


def check_eval(arm, rtag, rnum, seed, need_para1=True):
    """eval 파일 하나를 검사하고 (드리프트 목록, env)를 돌려준다.

    ★학습 유무로 요구 수준을 나눈다:
      무학습 arm  -> d == 0 을 **강제**한다. eval 파이프라인은 결정적이므로 어기면 진짜 이상이다.
      재학습 arm  -> d != 0 이면 **경고만** 하고 계속한다 (교훈 22).
    """
    ev = json.load(open(f"results/eval_{arm}_r{rtag}_s{seed}.json", encoding="utf-8"))
    if need_para1:
        assert "agreeable_para1" in ev["conditions"], (
            f"{arm}: agreeable_para1이 없다 — 이 배치의 핵심 셀이 비었다")
    meta = KNOWN.get((arm, rnum), {})
    trained = meta.get("trained", True)
    counts = meta.get("counts", {})
    env = ev.get("env") or {}
    print(f"  {arm} s{seed}  [{'재학습' if trained else '학습 없음'}]"
          f"  env={env.get('unsloth', '?')}/torch {env.get('torch', '?')}")
    moved = []
    for lens, c in ev["conditions"].items():
        assert len(c["records"]) == N, f"{arm}/{lens}: n={len(c['records'])}"
        n_ref = round(c["refusal_rate"] * N)
        exp = counts.get(lens)
        mark = ""
        if exp is not None:
            d = n_ref - exp
            mark = " (기존과 일치)" if d == 0 else f" ★기존 {exp}에서 {d:+d}"
            if d != 0:
                moved.append((arm, lens, d, trained))
        print(f"    {lens:>16} refusal={c['refusal_rate']:.3f} "
              f"({n_ref}/{N}) harm={c['harm_score']:.3f}{mark}")
    # 무학습 arm은 정확 재현이 **요구사항**이다
    if not trained:
        bad = [(l, d) for a, l, d, t in moved if not t]
        assert not bad, (
            f"{arm}은 학습이 없는데 {bad}만큼 달라졌다 — 생성은 temperature=0.0으로 결정적이다. "
            f"eval 파이프라인이나 프롬프트 집합이 바뀌었다는 뜻이므로 결과를 믿지 마라")
    return moved, env


if BATCH == 1:
    drift = []
    for name, arm, seed in PLAN:
        rtag = "0" if arm == "arm0" else "3"
        rnum = 0.0 if arm == "arm0" else 3.0
        moved, _ = check_eval(arm, rtag, rnum, seed)
        drift += moved

    if drift:
        print(f"\n  ⚠️재학습 arm에서 {len(drift)}개 렌즈가 정확히 재현되지 않았다:")
        for arm, lens, d, _ in drift:
            print(f"     {arm}/{lens}: {d:+d}")
        print("     같은 시드라도 라이브러리 버전이 바뀌면 체크포인트가 달라진다(교훈 22).")
        print("     위 env 줄을 기존 산출물의 env와 대조할 것. **실패가 아니다.**")
    else:
        print("\n  [ok] 기존 렌즈가 전부 개수까지 정확히 재현됐다")

    kr, kh = read_probe("arm2", "3", 42)
    print(f"  probe arm2 s42 kappa_v_ref={kr:.4f} kappa_v_harm={kh:.4f}")

    a0 = json.load(open("results/eval_arm0_r0_s42.json", encoding="utf-8"))["conditions"]
    print(f"\n  ★판정: arm0 none={a0['none']['refusal_rate']:.3f} "
          f"agreeable={a0['agreeable']['refusal_rate']:.3f} "
          f"para1={a0['agreeable_para1']['refusal_rate']:.3f}")
    print("     para1이 none과 비슷하면 → 패러프레이즈 결과가 더 강해진다")
    print("     para1이 크게 낮으면    → 효과의 일부는 프롬프트 세기다. 주장 범위를 좁혀라")

else:
    b = json.load(open("results/probe_arm0_r0_s42.json", encoding="utf-8"))
    assert b.get("schema") == "probe_r3", "동결기저가 아직 옛 스키마다"
    b_ref = b["readings"]["kappa_v_ref"]
    b_harm = b["readings"]["kappa_v_harm"]
    print(f"  동결기저 arm0 r0: kappa_v_ref={b_ref:.4f} kappa_v_harm={b_harm:.4f} "
          f"(차 {b_harm - b_ref:+.4f})")

    print("\n  기하 궤적 — ★간극(kappa_harm − kappa_ref)이 라운드에 따라 움직이는가:")
    for name, arm, seed in PLAN:
        for r in (1, 2, 3):
            kr, kh = read_probe(arm, str(r), seed)
            print(f"    {arm} s{seed} r{r}  kappa_v_ref={kr:.4f} kappa_v_harm={kh:.4f} "
                  f"차={kh - kr:+.4f}")

    print("\n  r3 eval 재측정 (궤적과 **같은 체크포인트**여야 비교가 성립한다):")
    drift = []
    for name, arm, seed in PLAN:
        moved, _ = check_eval(arm, "3", 3.0, seed)
        drift += moved
    if drift:
        print(f"\n  ⚠️재학습이라 {len(drift)}개 렌즈가 기존과 다르다 — 예상된 것이다(교훈 22).")
        print("     ★이 배치의 r3 eval이 **새 정본**이다. 궤적 프로브와 같은 모델에서 나왔기 때문.")
    else:
        print("\n  [ok] r3 eval이 기존과 정확히 일치 — 환경이 안 바뀌었다")

    print("\n  ⚠️★이 배치는 eval_arm{1,3}_r3_s42.json을 바꿨다 ⇒ **EXP-M03 표본이 무효다.**")
    print("     로컬에서 `python src/sample_m03.py --seed 42 --force`로 다시 뽑을 것.")

print(f"\n[gate] R4 BATCH {BATCH} 통과 — {len(PLAN)}세션")
